# Academic Advisor RAG – Exploration Notebook

Use this notebook to test each component independently before running the full API.

In [ ]:
import sys, os
sys.path.insert(0, '..')  # project root

# Load .env
from dotenv import load_dotenv
load_dotenv('../.env')
print('API key set:', bool(os.environ.get('OPENAI_API_KEY')))

## 1. PDF Extraction – pdfplumber

In [ ]:
from app.pdf_processor import PDFProcessor

processor = PDFProcessor('../data/handbook.pdf')
docs = processor.extract(use_tabula=True, use_camelot=False)

print(f'Total documents: {len(docs)}')
print('\n--- Sample (page 35) ---')
sample = [d for d in docs if d.metadata.get('page') == 35]
if sample:
    print(sample[0].page_content[:600])

## 2. Text Splitting

In [ ]:
from app.vector_store import get_text_splitter

splitter = get_text_splitter()
chunks = splitter.split_documents(docs[:20])
print(f'Chunks from first 20 docs: {len(chunks)}')
print(f'Avg chunk length: {sum(len(c.page_content) for c in chunks)//len(chunks)} chars')
print('\n--- Sample chunk ---')
print(chunks[5].page_content)

## 3. Vector Store – Build (runs embeddings, costs tokens)

In [ ]:
# ⚠️ This will call OpenAI embeddings API
from app.vector_store import get_vector_store_manager

vsm = get_vector_store_manager()
# vsm.build_all(docs)   # uncomment to rebuild
print('Vector store ready:', vsm.is_ready)

## 4. Retrieval Test

In [ ]:
retriever = vsm.get_retriever(k=5)
query = 'What are the prerequisites for Data Structures?'
results = retriever.invoke(query)
print(f'Retrieved {len(results)} chunks for: "{query}"\n')
for i, doc in enumerate(results, 1):
    print(f'[{i}] p.{doc.metadata.get("page","?")} – {doc.page_content[:200]}')
    print()

## 5. Student Profile – Course Recommendation

In [ ]:
import asyncio
from app.models import StudentProfile, Program, AcademicLevel, Semester
from app.rag_chain import recommend_courses

profile = StudentProfile(
    name='Ahmed Ali',
    program=Program.CS,
    level=AcademicLevel.SOPHOMORE,
    cgpa=2.8,
    earned_hours=36,
    current_semester=Semester.FALL,
    passed_courses=[
        'ENG101','BMT111','BMT112','BPH111','CSC111','PSC110',
        'ENG102','BMT121','CSC121','CPS121','BMT122','CSC122',
        'CPS211','CSC211','BPH211','BMT212','BMT211',
    ],
    currently_enrolled=['CSC221','CSC222'],
    failed_courses=[],
)

result = asyncio.run(recommend_courses(
    session_id='test-session-001',
    question='What courses can I register for this semester?',
    profile=profile,
))
print(result['answer'])

## 6. General Q&A Chain

In [ ]:
from app.rag_chain import answer_question

result = asyncio.run(answer_question(
    session_id='test-session-002',
    question='How many credit hours can I take if my CGPA is 1.8?',
))
print(result['answer'])
print('\nSources:', result['sources'])